In [1]:
import pandas as pd
from rdkit import Chem, RDLogger
df_chemistry = pd.read_pickle(
    "/home/susan/mof-co2-adsorption/data/processed/hmof_linker_extracted.pkl"
)
df_chemistry.columns.tolist()


['filename',
 'lcd',
 'pld',
 'void_fraction',
 'surface_area_m2g',
 'mofid',
 'CO2_uptake_0.01bar_molkg',
 'CO2_uptake_0.05bar_molkg',
 'CO2_uptake_0.1bar_molkg',
 'CO2_uptake_0.5bar_molkg',
 'CO2_uptake_2.5bar_molkg',
 'chemical_representation',
 'linker_smiles',
 'invalid_fragments',
 'linker_1',
 'linker_2',
 'linker_3',
 'linker_4',
 'linker_5',
 'linker_6',
 'linker_7',
 'linker_8',
 'linker_9',
 'linker_10',
 'linker_11',
 'linker_12',
 'linker_13']

In [2]:
linker_columns = [f"linker_{i}" for i in range(1, 14)]

df_chemistry[linker_columns].head()

,linker_1,linker_2,linker_3,linker_4,linker_5,linker_6,linker_7,linker_8,linker_9,linker_10,linker_11,linker_12,linker_13
0,O=C([O-])c1ccc(C(=O)[O-])cc1,None,None,None,None,None,None,None,None,None,None,None,None
1,O=C([O-])c1ccc(C(=O)[O-])cc1,None,None,None,None,None,None,None,None,None,None,None,None
2,O=C([O-])c1cc(F)c(C(=O)[O-])c(F)c1F,None,None,None,None,None,None,None,None,None,None,None,None
3,COc1cc(C(=O)[O-])cc(OC)c1C(=O)[O-],COc1cc(C(=O)[O-])ccc1C(=O)[O-],None,None,None,None,None,None,None,None,None,None,None
4,CCc1cc(C(=O)[O-])c(CC)c(CC)c1C(=O)[O-],O=C([O-])C#CC(=O)[O-],None,None,None,None,None,None,None,None,None,None,None


| Descriptor         | Meaning                        |
| ------------------ | ------------------------------ |
| `MolWt`            | Molecular weight               |
| `TPSA`             | Topological polar surface area |
| `NumHAcceptors`    | Hydrogen-bond acceptor count   |
| `NumHDonors`       | Hydrogen-bond donor count      |
| `NumAromaticRings` | Aromatic ring count            |


In [3]:
from rdkit.Chem import Descriptors

# 1. Select the first MOF's first linker
smiles = df_chemistry["linker_1"].iloc[0]

# 2. Convert its SMILES into an RDKit molecule
mol = Chem.MolFromSmiles(smiles)

# 3. Calculate a small starting set of descriptors
if mol is None:
    print("RDKit could not read this SMILES:", smiles)
else:
    linker_descriptors = {
        "MolWt": Descriptors.MolWt(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumAromaticRings": Descriptors.NumAromaticRings(mol),
    }

    display(pd.DataFrame([linker_descriptors]))

,MolWt,TPSA,NumHAcceptors,NumHDonors,NumAromaticRings
0,164.116,80.26,4,0,1


| Column type           | Missing value |
| --------------------- | ------------- |
| Linker SMILES         | `None`        |
| Numerical descriptors | `NaN`         |


| Your features                          | What they capture—and how to interpret them                                                                                                            |
| -------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `MaxPartialCharge`, `MinPartialCharge` | Extremes of estimated atomic charge. For your charged, extracted linkers, treat these as approximate features—not actual charges in the assembled MOF. |
| `TPSA`                                 | Linker polar surface area; already in your original five.                                                                                              |
| `NOCount`                              | Combined nitrogen and oxygen count—not the number of accessible CO₂-binding sites.                                                                     |
| `NumHeteroatoms`                       | Number of atoms other than carbon and hydrogen.                                                                                                        |
| `MolLogP`                              | Estimated octanol/water partitioning; an exploratory feature, especially for charged linkers. It does not directly measure CO₂ affinity.               |
| `LabuteASA`                            | Approximate molecular surface area of the linker—not the MOF’s accessible surface area.                                                                |
| `NumRotatableBonds`, `FractionCSP3`    | Indicators of linker flexibility and carbon hybridization—not direct measurements of framework rigidity.                                               |


In [4]:
def calculate_all_descriptors(smiles):
    missing_descriptors = {
        "MolWt": float("nan"),
        "TPSA": float("nan"),
        "NumHAcceptors": float("nan"),
        "NumHDonors": float("nan"),
        "NumAromaticRings": float("nan"),
        "MaxPartialCharge": float("nan"),
        "MinPartialCharge": float("nan"),
        "NOCount": float("nan"),
        "NumHeteroatoms": float("nan"),
        "LogP": float("nan"),
        "LabuteASA": float("nan"),
        "RotatableBonds": float("nan"),
        "FractionCSP3": float("nan"),
        "PEOE_VSA1": float("nan"),
        "PEOE_VSA2": float("nan"),
    }

    if pd.isna(smiles) or smiles.strip() == "":
        return missing_descriptors

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print("RDKit could not read this SMILES:", smiles)
        return missing_descriptors

    return {
        "MolWt": Descriptors.MolWt(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumAromaticRings": Descriptors.NumAromaticRings(mol),
        "MaxPartialCharge": Descriptors.MaxPartialCharge(mol),
        "MinPartialCharge": Descriptors.MinPartialCharge(mol),
        "NOCount": Descriptors.NOCount(mol),
        "NumHeteroatoms": Descriptors.NumHeteroatoms(mol),
        "LogP": Descriptors.MolLogP(mol),
        "LabuteASA": Descriptors.LabuteASA(mol),
        "RotatableBonds": Descriptors.NumRotatableBonds(mol),
        "FractionCSP3": Descriptors.FractionCSP3(mol),
        "PEOE_VSA1": Descriptors.PEOE_VSA1(mol),
        "PEOE_VSA2": Descriptors.PEOE_VSA2(mol),}

In [5]:
# Testing the Function : 
calculate_all_descriptors(df_chemistry["linker_1"].iloc[0])

{'MolWt': 164.11599999999999,
 'TPSA': 80.25999999999999,
 'NumHAcceptors': 4,
 'NumHDonors': 0,
 'NumAromaticRings': 1,
 'MaxPartialCharge': 0.07148182559135696,
 'MinPartialCharge': -0.5448151129346132,
 'NOCount': 4,
 'NumHeteroatoms': 4,
 'LogP': -1.5864000000000003,
 'LabuteASA': 68.07279905276974,
 'RotatableBonds': 2,
 'FractionCSP3': 0.0,
 'PEOE_VSA1': 19.802129157825057,
 'PEOE_VSA2': 0.0}

In [8]:
linker_columns = [f"linker_{i}" for i in range(1, 14)]

descriptor_tables = []

for column in linker_columns:
    # Calculate descriptors for every SMILES in this column
    results = df_chemistry[column].apply(calculate_all_descriptors)

    # Create a descriptor table with the original row index
    descriptor_df = pd.DataFrame(
        results.tolist(),
        index=df_chemistry.index
    )

    # Label descriptors with their linker position
    descriptor_df = descriptor_df.add_prefix(f"{column}_")

    # Collect tables instead of inserting columns into df_chemistry
    descriptor_tables.append(descriptor_df)

# Combine all 13 descriptor tables
all_linker_descriptors = pd.concat(descriptor_tables, axis=1)

# Remove previous versions of these columns so rerunning creates no duplicates
df_chemistry = df_chemistry.drop(
    columns=all_linker_descriptors.columns,
    errors="ignore"
)

# Add all descriptor columns at once
df_chemistry = pd.concat(
    [df_chemistry, all_linker_descriptors],
    axis=1
).copy()

print("Descriptor columns:", all_linker_descriptors.shape[1])  # Expected: 195
print("Dataset shape:", df_chemistry.shape)



Descriptor columns: 195
Dataset shape: (25929, 222)


In [10]:
# To check one MOF and all 15 descriptors for linker_1,
columns_to_check = ["linker_1"] + [
    col for col in df_chemistry.columns
    if col.startswith("linker_1_")
]

df_chemistry[columns_to_check].iloc[0]

linker_1                     O=C([O-])c1ccc(C(=O)[O-])cc1
linker_1_MolWt                                    164.116
linker_1_TPSA                                       80.26
linker_1_NumHAcceptors                                  4
linker_1_NumHDonors                                     0
linker_1_NumAromaticRings                               1
linker_1_MaxPartialCharge                        0.071482
linker_1_MinPartialCharge                       -0.544815
linker_1_NOCount                                        4
linker_1_NumHeteroatoms                                 4
linker_1_LogP                                     -1.5864
linker_1_LabuteASA                              68.072799
linker_1_RotatableBonds                                 2
linker_1_FractionCSP3                                 0.0
linker_1_PEOE_VSA1                              19.802129
linker_1_PEOE_VSA2                                    0.0
Name: 0, dtype: object